# PHYS690 — Lecture 11: Finite Differences, Boundary Conditions, and Grids
## Electrostatic field solver from a boundary-value problem

**Primary physics example:** a two-dimensional electrostatic potential $V(x,y)$ in a rectangular region with conducting boundaries.

Lecture 10 turned an initial-value differential equation into a sequence of time steps. Today we take a different route: replace spatial derivatives by finite differences so that an electrostatic boundary-value problem becomes a **sparse linear algebra problem**.

This is the same mathematical structure that appears in graduate E&M when using separation of variables, Green functions, uniqueness theorems, image charges, and numerical field maps. The computational question is: how do we turn the boundary conditions and charge density into a reproducible field solver?

## Learning Goals

By the end of class you should be able to:

- connect electrostatic boundary-value problems to Poisson's equation;
- derive the centered finite-difference stencil for the 2D Laplacian;
- explain how a 2D grid of unknown potentials becomes a 1D linear system;
- assemble and inspect a sparse matrix for a Dirichlet boundary-value problem;
- describe what `scipy.sparse` stores and what `scipy.sparse.linalg.spsolve` solves;
- compute the electric field from the numerical potential;
- distinguish a small **linear-system residual** from a small **discretization error**;
- measure grid convergence using an analytic E&M benchmark.

In [ ]:
# Core numerical array and table tools used throughout the lecture.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scipy.sparse stores matrices by their nonzero entries instead of as dense arrays.
from scipy import sparse

# sparse.linalg contains sparse linear-system solvers such as spsolve(A, b).
from scipy.sparse import linalg as spla

# Keep plots readable and visually consistent with the rest of the course.
plt.rcParams.update({"figure.figsize": (7, 4.8), "axes.grid": True})

## 1. Physics Model: Electrostatics as a PDE

In electrostatics,

$$
\nabla\cdot \mathbf E = \frac{\rho}{\epsilon_0},
\qquad
\mathbf E = -\nabla V.
$$

Combining these equations gives Poisson's equation for the potential:

$$
\boxed{-\nabla^2 V = \frac{\rho}{\epsilon_0}.}
$$

In a charge-free region, $\rho=0$, this reduces to Laplace's equation:

$$
\nabla^2 V=0.
$$

Boundary conditions are part of the physics model. For electrostatics, a conducting surface is an equipotential surface, so fixed-potential conductors are naturally represented by **Dirichlet boundary conditions**:

$$
V\big|_{\partial \Omega}=V_\mathrm{boundary}.
$$

The uniqueness theorem from E&M says that, for a well-posed Dirichlet problem, the potential inside the region is determined uniquely by the charge density and boundary values. Numerically, that means the boundary rows and source terms are part of the physics model, not just programming bookkeeping.

### Analytic Benchmark from Separation of Variables

To validate the solver, use a rectangular region $0\le x\le L_x$, $0\le y\le L_y$ with no charge density and boundary values

$$
V(0,y)=0,\qquad V(L_x,y)=0,\qquad V(x,0)=0,
$$

and

$$
V(x,L_y)=V_0\sin\left(\frac{\pi x}{L_x}\right).
$$

Separation of variables gives

$$
\boxed{
V(x,y)=V_0\sin\left(\frac{\pi x}{L_x}\right)
\frac{\sinh(\pi y/L_x)}{\sinh(\pi L_y/L_x)}.
}
$$

This is a useful E&M benchmark because it tests the boundary values, the finite-difference stencil, the sparse solve, and the field calculation against a known solution.

In [ ]:
# Geometry and scale. We use dimensionless electrostatic units for the benchmark.
# The computational method does not depend on this particular choice of units.
Lx = 1.0
Ly = 1.0
V0 = 1.0
eps0 = 1.0  # arbritrary units for the vacuum permittivity for simplicity in the benchmark problem


def potential_exact(X, Y, Lx=Lx, Ly=Ly, V0=V0):
    """Analytic Laplace-equation solution for validation, not assembly."""
    # This expression is the separation-of-variables benchmark in the notebook text.
    # We use it only for plotting and error checks after the numerical solve.
    return (
        V0
        * np.sin(np.pi * X / Lx)
        * np.sinh(np.pi * Y / Lx)
        / np.sinh(np.pi * Ly / Lx)
    )


def boundary_exact(X, Y, Lx=Lx, Ly=Ly, V0=V0):
    """Dirichlet boundary values for the rectangular benchmark.

    This function encodes only the boundary conditions, not the analytic
    solution in the interior. The interior entries are placeholders that will
    be overwritten by the numerical solve.
    """
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)

    # Start with the grounded boundaries: V=0 on left, right, and bottom.
    values = np.zeros_like(X, dtype=float)

    # The only nonzero boundary is the top conducting surface.
    top_boundary = np.isclose(Y, Ly)
    values[top_boundary] = V0 * np.sin(np.pi * X[top_boundary] / Lx)
    return values


def zero_charge_density(x, y):
    """Charge density rho(x,y) for the benchmark Laplace problem."""
    # rho=0 makes this a Laplace-equation test problem.
    return 0.0


def make_grid(nx, ny, Lx=Lx, Ly=Ly):
    """Return 1D coordinate arrays and 2D coordinate grids.

    With indexing="xy", X[j,i] and Y[j,i] give the coordinates of grid node
    (i,j). This matches the plotting convention used by contourf.
    """
    x = np.linspace(0.0, Lx, nx)
    y = np.linspace(0.0, Ly, ny)
    X, Y = np.meshgrid(x, y, indexing="xy")
    return x, y, X, Y

### Reference Plot: Analytic Potential

Before building the numerical solver, it helps to look at the benchmark solution we are trying to reproduce. The top boundary is held at $V_0\sin(\pi x/L_x)$, while the other three sides are grounded. The potential is largest near the top boundary and decays into the interior.

In [ ]:
# Plot the analytic separation-of-variables solution for reference.
# This plot is a target for the numerical solver, not an input to it.
x_ref, y_ref, X_ref, Y_ref = make_grid(101, 101)
V_exact_ref_plot = potential_exact(X_ref, Y_ref)

fig, ax = plt.subplots(figsize=(6.2, 5.2))
levels = np.linspace(0.0, V0, 25)

# Filled contours show the potential; black contour lines emphasize equipotentials.
contour = ax.contourf(X_ref, Y_ref, V_exact_ref_plot, levels=levels)
ax.contour(X_ref, Y_ref, V_exact_ref_plot, levels=levels[::4], colors="k", linewidths=0.5)
ax.set(
    xlabel="x",
    ylabel="y",
    title="Analytic benchmark potential for the rectangular Dirichlet problem",
    aspect="equal",
)
fig.colorbar(contour, ax=ax, label="V")
plt.show()

## 2. Grid and Centered Finite Difference

Use a rectangular grid with $n_x$ points in the $x$ direction and $n_y$ points in the $y$ direction:

$$
x_i=i\Delta x,
\qquad
y_j=j\Delta y.
$$

At an interior grid point $(x_i,y_j)$, the centered finite-difference approximation to the Laplacian is

$$
\nabla^2 V(x_i,y_j)
\approx
\frac{V_{i+1,j}-2V_{i,j}+V_{i-1,j}}{\Delta x^2}
+
\frac{V_{i,j+1}-2V_{i,j}+V_{i,j-1}}{\Delta y^2}.
$$

For Poisson's equation in the form $-\nabla^2V=\rho/\epsilon_0$, this becomes

$$
\left(\frac{2}{\Delta x^2}+\frac{2}{\Delta y^2}\right)V_{i,j}
-\frac{V_{i-1,j}+V_{i+1,j}}{\Delta x^2}
-\frac{V_{i,j-1}+V_{i,j+1}}{\Delta y^2}
=\frac{\rho_{i,j}}{\epsilon_0}.
$$

This is one algebraic equation for one unknown potential value. Applying the same equation at every interior grid point gives a coupled linear system,

$$
\boxed{A\mathbf u = \mathbf b.}
$$

Here $\mathbf u$ is the vector of unknown interior potential values,

$$
\mathbf u=(V_{1,1},V_{2,1},\ldots,V_{n_x-2,1},V_{1,2},\ldots)^T.
$$

The matrix $A$ contains the finite-difference coefficients from the five-point stencil. Each row of $A$ corresponds to one interior grid point. The vector $\mathbf b$ contains the charge-density term $\rho_{i,j}/\epsilon_0$ plus any contributions from known Dirichlet boundary values.

Boundary values are not solved for. When a neighboring point is a known boundary value, its stencil contribution is moved to the right-hand side $\mathbf b$. Section 3 will show how this bookkeeping becomes a sparse matrix in code.

### Diagnostic Figure: What Are Boundary and Interior Grid Points?

The finite-difference equation is applied only at **interior** grid points. Boundary grid points already have prescribed values from the conducting surfaces. The indices $(i,j)$ label the grid point at $x_i=i\Delta x$ and $y_j=j\Delta y$.

The highlighted point below is one interior node. Its five-point stencil uses that node plus its left, right, bottom, and top neighbors.

In [ ]:
# Small orientation grid for discussing indices and boundary points.
# This grid is intentionally tiny so every label can be read on the figure.
orient_nx, orient_ny = 7, 6
orient_x, orient_y, orient_X, orient_Y = make_grid(orient_nx, orient_ny)

# Mark interior nodes with 1 and boundary nodes with 0 for the background color.
interior = np.ones((orient_ny, orient_nx), dtype=float)
interior[0, :] = 0.0
interior[-1, :] = 0.0
interior[:, 0] = 0.0
interior[:, -1] = 0.0

# Choose one interior point to track through the rest of the sparse-matrix examples.
selected_i, selected_j = 3, 2
neighbors = [
    (selected_i, selected_j),      # stencil center
    (selected_i - 1, selected_j),  # left neighbor
    (selected_i + 1, selected_j),  # right neighbor
    (selected_i, selected_j - 1),  # bottom neighbor
    (selected_i, selected_j + 1),  # top neighbor
]

fig, ax = plt.subplots(figsize=(7.5, 5.0))
ax.imshow(interior, origin="lower", cmap="Blues", vmin=0.0, vmax=1.0)

# Label boundary nodes as B and interior nodes by their grid index (i,j).
for j in range(orient_ny):
    for i in range(orient_nx):
        is_boundary = i == 0 or i == orient_nx - 1 or j == 0 or j == orient_ny - 1
        if is_boundary:
            label = "B"
            color = "black"
        else:
            label = f"({i},{j})"
            color = "white"
        ax.text(i, j, label, ha="center", va="center", fontsize=8, color=color, weight="bold")

# Mark the selected stencil point and its nearest neighbors.
for i, j in neighbors:
    ax.plot(i, j, "o", color="crimson", ms=10, fillstyle="none", mew=2)
ax.plot(selected_i, selected_j, "o", color="crimson", ms=6)
ax.text(
    selected_i + 0.25,
    selected_j + 0.25,
    "stencil center",
    color="crimson",
    fontsize=9,
    weight="bold",
)

ax.set(
    xticks=np.arange(orient_nx),
    yticks=np.arange(orient_ny),
    xlabel="grid index i",
    ylabel="grid index j",
    title="Boundary nodes, interior unknowns, and one five-point stencil",
)
ax.set_xticks(np.arange(-0.5, orient_nx, 1), minor=True)
ax.set_yticks(np.arange(-0.5, orient_ny, 1), minor=True)
ax.grid(which="minor", color="black", linewidth=0.8)
ax.tick_params(which="minor", bottom=False, left=False)
plt.show()

## 3. Production Pattern: Sparse Matrix Assembly

Lecture 10 used the damped oscillator to separate three ideas:

1. write the continuous equation;
2. choose a numerical approximation for the derivative or integral;
3. use a production solver only after understanding what problem the solver is solving.

The same pattern applies here. The continuous electrostatic problem is Poisson's equation. The finite-difference stencil turns that PDE into algebraic equations. The production solver `spsolve` then solves those algebraic equations.

For even a modest 2D grid, the matrix has many more zeros than nonzeros. A dense matrix stores every entry, including all the zeros. A sparse matrix stores only the nonzero entries and enough row/column information to know where they belong. For a 2D five-point Laplacian, each interior grid point couples only to itself and its nearest neighbors, so each row has at most five nonzero entries. That is exactly the kind of structure sparse matrices are designed for.

SciPy provides several sparse matrix formats. In this lecture we use two:

| Format | Why it appears here |
|---|---|
| `lil_matrix` | convenient while assembling entries row by row |
| `csr_matrix` | CSR means **Compressed Sparse Row**; efficient for arithmetic and sparse linear solves |

The workflow is:

1. assemble the matrix in a format that is easy to modify;
2. convert it to CSR (**Compressed Sparse Row**) with `.tocsr()`;
3. solve the sparse linear system with `scipy.sparse.linalg.spsolve`.

`spsolve(A, b)` does **not** form $A^{-1}$. It factors the sparse matrix internally and solves $A\mathbf u=\mathbf b$ using sparse linear algebra. That distinction matters because explicitly forming an inverse is usually slower, less accurate numerically, and much denser than the original matrix.

### Turning a 2D Grid into a 1D Unknown Vector

A matrix equation needs a one-dimensional unknown vector, but the physical problem lives on a two-dimensional grid. We choose a bookkeeping convention: order the unknowns row by row,

$$
\mathbf u = (V_{1,1},V_{2,1},\ldots,V_{n_x-2,1},V_{1,2},\ldots)^T.
$$

Only interior grid points are unknown for the benchmark Dirichlet problem. Boundary values are already known from the conducting surfaces and move to the right-hand side vector $\mathbf b$.

The helper `interior_index(i, j, nx)` converts a two-dimensional grid index into a one-dimensional row/column index in that unknown vector. This is like Lecture 10's choice to store the oscillator state as `[y, v]`: the physics has its own natural variables, but the solver needs a concrete array layout.

In [ ]:
def interior_index(i, j, nx):
    """Map interior grid index (i,j) to a 1D unknown-vector index.

    Boundary nodes are not included in the unknown vector. The first interior
    row has j=1 and contains i from 1 through nx-2.
    """
    return (j - 1) * (nx - 2) + (i - 1)


def build_dirichlet_poisson_system(
    nx,
    ny,
    rho_func=zero_charge_density,
    boundary_func=boundary_exact,
    Lx=Lx,
    Ly=Ly,
    eps0=eps0,
):
    """Build sparse A u = b for -laplacian(V) = rho/eps0 with Dirichlet BCs.

    Parameters
    ----------
    nx, ny : int
        Number of grid points in the x and y directions, including boundaries.
    rho_func : callable
        Charge density function rho(x,y). For the benchmark this is zero.
    boundary_func : callable
        Function returning prescribed V(x,y) values on the boundary.
    """
    if nx < 3 or ny < 3:
        raise ValueError("nx and ny must each be at least 3.")

    # Build the physical grid and spacing. Unknowns live only on x[1:-1], y[1:-1].
    x, y, X, Y = make_grid(nx, ny, Lx=Lx, Ly=Ly)
    dx = x[1] - x[0]
    dy = y[1] - y[0]

    # The boundary function only encodes known Dirichlet values. Interior
    # entries are placeholders and are overwritten after solving.
    boundary_values = np.asarray(boundary_func(X, Y), dtype=float)

    # There is one unknown for each interior grid node.
    n_unknown = (nx - 2) * (ny - 2)

    # LIL format is convenient for row-by-row assembly. We convert to CSR later.
    A = sparse.lil_matrix((n_unknown, n_unknown), dtype=float)
    b = np.zeros(n_unknown, dtype=float)

    for j in range(1, ny - 1):
        for i in range(1, nx - 1):
            row = interior_index(i, j, nx)

            # Diagonal coefficient for V_{i,j} from the five-point stencil.
            A[row, row] = 2.0 / dx**2 + 2.0 / dy**2

            # Source term for -laplacian(V)=rho/eps0.
            b[row] = rho_func(x[i], y[j]) / eps0

            # Left and right neighbors. Unknown neighbors become matrix entries;
            # boundary neighbors are known values and move to the RHS.
            for neighbor_i, coeff in [(i - 1, -1.0 / dx**2), (i + 1, -1.0 / dx**2)]:
                if neighbor_i == 0 or neighbor_i == nx - 1:
                    b[row] -= coeff * boundary_values[j, neighbor_i]
                else:
                    col = interior_index(neighbor_i, j, nx)
                    A[row, col] = coeff

            # Bottom and top neighbors use the same logic in the y direction.
            for neighbor_j, coeff in [(j - 1, -1.0 / dy**2), (j + 1, -1.0 / dy**2)]:
                if neighbor_j == 0 or neighbor_j == ny - 1:
                    b[row] -= coeff * boundary_values[neighbor_j, i]
                else:
                    col = interior_index(i, neighbor_j, nx)
                    A[row, col] = coeff

    grid = {"x": x, "y": y, "X": X, "Y": Y, "dx": dx, "dy": dy}

    # CSR is the efficient sparse format for arithmetic and sparse solves.
    return grid, A.tocsr(), b, boundary_values


def solve_electrostatic_dirichlet(nx, ny):
    """Solve the rectangular benchmark and return grid, potential, matrix, and RHS."""
    grid, A, b, boundary_values = build_dirichlet_poisson_system(nx, ny)

    # Solve only for the interior unknown vector u.
    interior_solution = spla.spsolve(A, b)

    # Reconstruct the full 2D potential array, including known boundary values.
    V = boundary_values.copy()
    V[1:-1, 1:-1] = interior_solution.reshape((ny - 2, nx - 2))

    return {
        "grid": grid,
        "V": V,
        "A": A,
        "b": b,
        "u": interior_solution,
        "boundary_values": boundary_values,
    }

### Diagnostic Figure: 2D Grid Indices to 1D Unknown Indices

Use the **same toy grid and the same highlighted stencil** from Section 2. Boundary nodes are still labeled `B`, but now each interior node is labeled by its one-dimensional unknown-vector index. This shows exactly how the physical grid becomes the vector $\mathbf u$ in $A\mathbf u=\mathbf b$.

The crimson stencil is centered on the same $(i,j)$ point as before. In the matrix, the center point becomes one row index, and its neighboring unknowns become nonzero column entries in that row.

In [ ]:
# Reuse the same orientation grid and highlighted stencil from Section 2.
demo_nx, demo_ny = orient_nx, orient_ny

# index_map[j,i] is -1 on boundary nodes and the unknown-vector index inside.
index_map = -np.ones((demo_ny, demo_nx), dtype=int)
for j in range(1, demo_ny - 1):
    for i in range(1, demo_nx - 1):
        index_map[j, i] = interior_index(i, j, demo_nx)

fig, ax = plt.subplots(figsize=(7.5, 5.0))
cell_type = (index_map >= 0).astype(float)
ax.imshow(cell_type, origin="lower", cmap="Blues", vmin=0, vmax=1)

# Relabel interior nodes by row/column index in the vector u.
for j in range(demo_ny):
    for i in range(demo_nx):
        is_boundary = index_map[j, i] < 0
        label = "B" if is_boundary else str(index_map[j, i])
        color = "black" if is_boundary else "white"
        ax.text(i, j, label, ha="center", va="center", color=color, fontsize=9, weight="bold")

# Carry over the same stencil highlight from the Section 2 orientation plot.
for i, j in neighbors:
    ax.plot(i, j, "o", color="crimson", ms=10, fillstyle="none", mew=2)
ax.plot(selected_i, selected_j, "o", color="crimson", ms=6)
selected_row = interior_index(selected_i, selected_j, demo_nx)
ax.text(
    selected_i + 0.25,
    selected_j + 0.25,
    f"row {selected_row}",
    color="crimson",
    fontsize=9,
    weight="bold",
)

ax.set(
    xticks=np.arange(demo_nx),
    yticks=np.arange(demo_ny),
    xlabel="grid index i",
    ylabel="grid index j",
    title="Same grid: interior nodes relabeled by 1D unknown-vector index",
)
ax.set_xticks(np.arange(-0.5, demo_nx, 1), minor=True)
ax.set_yticks(np.arange(-0.5, demo_ny, 1), minor=True)
ax.grid(which="minor", color="black", linewidth=0.8)
ax.tick_params(which="minor", bottom=False, left=False)
plt.show()

### Diagnostic Figure: Sparsity Pattern and One Matrix Row

Now use the same grid and the same highlighted stencil one more time. The sparse matrix records which unknowns talk to which other unknowns. For the five-point stencil, one interior grid point couples to itself and to its left, right, bottom, and top neighbors, unless one of those neighbors is a boundary value.

The left panel repeats the 1D unknown-index map so the selected row has a physical location. The middle panel shows the nonzero pattern of $A$, with the selected row highlighted. The right panel shows the actual coefficients in that row.

In [ ]:
# Build a tiny sparse system using the same grid and stencil highlighted above.
demo_grid, demo_A, demo_b, demo_boundary = build_dirichlet_poisson_system(demo_nx, demo_ny)

# Select the row in A corresponding to the highlighted grid point.
selected_row = interior_index(selected_i, selected_j, demo_nx)
row_dense = demo_A.getrow(selected_row).toarray().ravel()
nonzero_cols = np.flatnonzero(row_dense)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)

# Panel 1: repeat the grid-index map with the same stencil highlighted.
cell_type = (index_map >= 0).astype(float)
axes[0].imshow(cell_type, origin="lower", cmap="Blues", vmin=0, vmax=1)
for j in range(demo_ny):
    for i in range(demo_nx):
        is_boundary = index_map[j, i] < 0
        label = "B" if is_boundary else str(index_map[j, i])
        color = "black" if is_boundary else "white"
        axes[0].text(i, j, label, ha="center", va="center", color=color, fontsize=8, weight="bold")
for i, j in neighbors:
    axes[0].plot(i, j, "o", color="crimson", ms=9, fillstyle="none", mew=2)
axes[0].plot(selected_i, selected_j, "o", color="crimson", ms=5)
axes[0].set(
    xticks=np.arange(demo_nx),
    yticks=np.arange(demo_ny),
    xlabel="grid index i",
    ylabel="grid index j",
    title=f"selected stencil center: row {selected_row}",
)
axes[0].set_xticks(np.arange(-0.5, demo_nx, 1), minor=True)
axes[0].set_yticks(np.arange(-0.5, demo_ny, 1), minor=True)
axes[0].grid(which="minor", color="black", linewidth=0.8)
axes[0].tick_params(which="minor", bottom=False, left=False)

# Panel 2: spy(A) marks nonzero matrix entries. The horizontal line is the selected row.
axes[1].spy(demo_A, markersize=8)
axes[1].axhline(selected_row, color="crimson", linewidth=1.5)
axes[1].set_title("nonzero pattern of sparse A")
axes[1].set_xlabel("column index")
axes[1].set_ylabel("row index")

# Panel 3: show the coefficients in the selected row.
bar_colors = ["crimson" if col == selected_row else "steelblue" for col in nonzero_cols]
axes[2].bar(nonzero_cols, row_dense[nonzero_cols], color=bar_colors)
axes[2].set(
    xlabel="column index in unknown vector",
    ylabel="matrix coefficient",
    title=f"row {selected_row}: coefficients for (i={selected_i}, j={selected_j})",
)
axes[2].axhline(0.0, color="black", linewidth=0.8)
plt.show()

print(f"selected grid point (i,j) = ({selected_i}, {selected_j})")
print(f"selected row index = {selected_row}")
print(f"nonzero columns in this row = {nonzero_cols.tolist()}")
print(f"demo_A shape = {demo_A.shape}")
print(f"stored nonzero entries = {demo_A.nnz}")
print(f"dense entries would be = {demo_A.shape[0] * demo_A.shape[1]}")
print(f"fraction stored = {demo_A.nnz / (demo_A.shape[0] * demo_A.shape[1]):.3f}")

### Reading One Row of the Matrix as a Physics Equation

Each row of $A\mathbf u=\mathbf b$ is one finite-difference equation at one interior grid point.

For an interior point whose four neighbors are also unknowns, the row has the pattern

$$
\left(\frac{2}{\Delta x^2}+\frac{2}{\Delta y^2}\right)V_{i,j}
-\frac{1}{\Delta x^2}V_{i-1,j}
-\frac{1}{\Delta x^2}V_{i+1,j}
-\frac{1}{\Delta y^2}V_{i,j-1}
-\frac{1}{\Delta y^2}V_{i,j+1}
= \frac{\rho_{i,j}}{\epsilon_0}.
$$

Those five coefficients become five nonzero entries in one row of `A`.

If one neighbor is a fixed boundary value, it is not an unknown and therefore does not get a column in `A`. Instead, its contribution moves to the right-hand side. For example, if $V_{i,j+1}$ is known from the top boundary,

$$
-\frac{1}{\Delta y^2}V_{i,j+1}
$$

moves from the left side to the right side as

$$
+\frac{1}{\Delta y^2}V_\mathrm{boundary}.
$$

This is the PDE version of a familiar computational move from Lecture 10: first decide what variables the solver is responsible for, then put everything already known into the inputs or right-hand side.

### From Lecture 10's `solve_ivp` to Today's `spsolve`

The SciPy calls in Lectures 10 and 11 solve different mathematical problems, but the workflow is intentionally similar.

| Lecture 10 ODE workflow | Lecture 11 PDE/BVP workflow |
|---|---|
| Define a state vector, e.g. $\mathbf u=[y,v]$. | Define an unknown vector of interior potentials, $\mathbf u=(V_{1,1},V_{2,1},\ldots)^T$. |
| Write a function for the derivative, $d\mathbf u/dt=f(t,\mathbf u)$. | Write a stencil for the operator, $-\nabla^2 V=\rho/\epsilon_0$. |
| Use `solve_ivp` to advance the IVP in time. | Use `spsolve` to solve the sparse algebraic system once. |
| Validate with step-size studies and physical diagnostics. | Validate with grid-convergence studies, residuals, and field diagnostics. |

The main conceptual difference is that an ODE initial-value problem evolves from an initial state, while an elliptic PDE boundary-value problem solves for all interior grid values at once for a fixed point in time, constrained by the boundary conditions.

## In-class Activity 1 — Solve the Electrostatic Benchmark (14 Minutes)

Use the production solver above on a $41\times 41$ grid. Complete only the lines marked `TODO`.

Questions to keep in mind while coding:

- Where are the boundary values stored?
- Which array contains the full two-dimensional potential field?
- Does the numerical potential have the same qualitative shape as the separation-of-variables solution?

In [ ]:
# TODO: solve the benchmark problem on a 41 by 41 grid.
# The returned dictionary contains the grid, potential field, sparse matrix, RHS, and unknown vector.
solution_41 = ...

# TODO: extract the grid and potential from the returned dictionary.
grid_41 = ...
V_41 = ...

# TODO: evaluate the analytic solution on the same grid for validation.
# Use this only after the solve, not during matrix assembly.
V_exact_41 = ...

# TODO: compute the maximum absolute error over the full grid.
max_error_41 = ...
print(f"max |V_num - V_exact| = {max_error_41:.3e}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
# TODO: add filled contour plots of V_41 and V_exact_41.
# Hint: use grid_41["X"] and grid_41["Y"] for both panels.
...
plt.show()

## 4. Residual, Discretization Error, and Electric Field

Two diagnostics answer different questions.

### Algebraic residual

After solving $A\mathbf u=\mathbf b$, compute

$$
\mathbf r=A\mathbf u-\mathbf b.
$$

A small residual tells us the **discrete sparse linear system** was solved accurately.

### Discretization error

If an analytic benchmark is available, compute

$$
e_{i,j}=V_{i,j}^\mathrm{num}-V_\mathrm{exact}(x_i,y_j).
$$

A small residual does **not** imply a small discretization error. We can solve a coarse or incorrectly assembled finite-difference problem to machine precision.

### Electric field

Once $V$ is known, the electric field follows from

$$
\mathbf E = -\nabla V
= -\frac{\partial V}{\partial x}\hat{\mathbf x}
  -\frac{\partial V}{\partial y}\hat{\mathbf y}.
$$

Field plots are often the most physically useful output of an electrostatic solver, but the potential solve is the primary boundary-value problem.

In [ ]:
def electrostatic_diagnostics(solution):
    """Return residual and benchmark-error diagnostics for a completed solve."""
    grid = solution["grid"]
    V = solution["V"]
    A = solution["A"]
    b = solution["b"]
    u = solution["u"]

    # Algebraic residual: tests whether the sparse linear system was solved.
    residual = A @ u - b

    # Benchmark error: tests whether the discrete PDE approximates the continuous solution.
    exact = potential_exact(grid["X"], grid["Y"])
    error = V - exact

    return {
        "nx": grid["x"].size,
        "ny": grid["y"].size,
        "dx": grid["dx"],
        "dy": grid["dy"],
        "max_abs_error": np.max(np.abs(error)),
        "rms_error": np.sqrt(np.mean(error**2)),
        "residual_2norm": np.linalg.norm(residual),
        "relative_residual": np.linalg.norm(residual) / np.linalg.norm(b),
    }


def compute_electric_field(grid, V):
    """Compute Ex and Ey from V using finite differences."""
    x = grid["x"]
    y = grid["y"]

    # V has shape (ny, nx), so axis 0 is y and axis 1 is x.
    # np.gradient returns derivatives in that axis order.
    dV_dy, dV_dx = np.gradient(V, y, x, edge_order=2)
    Ex = -dV_dx
    Ey = -dV_dy
    return Ex, Ey


def plot_potential_and_field(grid, V, title="Electrostatic field map"):
    """Plot equipotentials and electric-field streamlines."""
    Ex, Ey = compute_electric_field(grid, V)
    X = grid["X"]
    Y = grid["Y"]

    fig, ax = plt.subplots(figsize=(6.2, 5.2))
    levels = np.linspace(np.nanmin(V), np.nanmax(V), 25)

    # Filled contours show V; line contours make equipotentials easier to see.
    contour = ax.contourf(X, Y, V, levels=levels)
    ax.contour(X, Y, V, levels=levels[::3], colors="k", linewidths=0.5, alpha=0.6)

    # Streamlines show the direction of E = -grad(V).
    ax.streamplot(grid["x"], grid["y"], Ex, Ey, color="white", density=1.0, linewidth=0.8)
    ax.set(xlabel="x", ylabel="y", title=title, aspect="equal")
    fig.colorbar(contour, ax=ax, label="V")
    plt.show()


# Use the solution produced in Activity 1. This keeps Section 4 connected to the activity.
diagnostics = electrostatic_diagnostics(solution_41)
pd.DataFrame([diagnostics])

In [ ]:
# Visual diagnostic for the same solution_41 from Activity 1.
# Equipotential contours should be perpendicular to electric-field streamlines.
plot_potential_and_field(solution_41["grid"], solution_41["V"], title="Benchmark potential and electric field")

### Discussion Prompt

The residual in the diagnostic table is much smaller than the potential error.

- What part of the calculation does the residual test?
- What part of the calculation does the benchmark error test?
- Why can a field map look physically plausible even when the grid is still too coarse?

## 5. Grid Convergence

The centered second-derivative stencil is second-order accurate for smooth solutions. For the potential error, we expect

$$
\epsilon(\Delta x)\approx C\Delta x^2
$$

when the grid is fine enough to be in the asymptotic regime. On a log-log plot, the slope of error versus grid spacing should approach 2.

This is one of the most important habits in numerical physics: do not only show a field map; show that the answer changes predictably when the grid is refined.

## In-class Activity 2 — Measure Grid Convergence (13 Minutes)

Use square grids with `n = 17, 25, 33, 49, 65`. For each grid:

1. solve the electrostatic benchmark;
2. compute the diagnostics;
3. append one row to a list;
4. build a pandas `DataFrame`;
5. estimate the observed slope of $\log(\epsilon)$ versus $\log(\Delta x)$.

In [ ]:
grid_sizes = [17, 25, 33, 49, 65]
rows = []

for n in grid_sizes:
    # TODO: solve the benchmark on an n by n grid.
    solution = ...

    # TODO: compute diagnostics for this grid and append one dictionary to rows.
    diagnostics = ...
    rows.append(diagnostics)

# The DataFrame should have one row per grid resolution.
convergence_df = pd.DataFrame(rows)
convergence_df

In [ ]:
fig, ax = plt.subplots()
# TODO: make a log-log plot of max_abs_error versus dx.
# Hint: ax.loglog(convergence_df["dx"], convergence_df["max_abs_error"], "o-")
...
ax.set(
    xlabel=r"grid spacing $\Delta x$",
    ylabel="maximum absolute error",
    title="Grid convergence of the electrostatic solver",
)
ax.legend()
plt.show()

In [ ]:
# TODO: compute the observed convergence slope from convergence_df.
# Hint: use np.polyfit() for log(max_abs_error) vs log(dx).
# The expected value is close to 2 for a second-order finite-difference stencil.
observed_order = ...
print(f"Observed finite-difference order: {observed_order:.3f}")

## 6. Common Failure Modes and Diagnostic Checks

**Boundary values omitted from the right-hand side**  
Dirichlet boundary values adjacent to unknown nodes enter $\mathbf b$. Forgetting that contribution changes the physics.

**Wrong sign in Poisson's equation**  
Decide whether you are solving $\nabla^2V=-\rho/\epsilon_0$ or $-\nabla^2V=\rho/\epsilon_0$, then make the matrix and right-hand side match.

**Confusing grid points with intervals**  
Here `nx` and `ny` are numbers of grid points, including boundary points. Then $\Delta x=L_x/(n_x-1)$.

**Small residual mistaken for physical accuracy**  
A residual tests the algebraic solve. Grid refinement tests the discretization.

**Dense matrices used for large PDEs**  
The 2D five-point stencil has only about five nonzero entries per row. Sparse storage and sparse solvers are essential once the grid becomes large.

**Indexing convention changed halfway through**  
If row-major indexing, column-major indexing, and array plotting conventions get mixed together, the matrix may still solve but represent the wrong geometry. Keep one mapping from `(i, j)` to row index and test it on a tiny grid.

## 7. Connection to Time-Dependent PDEs

Today solved an elliptic PDE: all grid points are coupled through one sparse linear system. Time-dependent PDEs use many of the same spatial stencils. For example, if a later calculation evolves a field $\psi(x,y,t)$, finite differences turn spatial derivatives into matrices or sparse stencil operations, while a time integrator advances the result.

The production pattern remains the same:

1. define the physics equation and boundary conditions;
2. choose a grid and discrete stencil;
3. assemble or apply the discrete operator;
4. solve the resulting algebraic problem;
5. validate with residuals, convergence, and physics checks.

## 8. Connection to Particle-in-Cell Codes in Plasma Physics

The electrostatic solve in this lecture is one piece of the algorithmic structure used in particle-in-cell (PIC) plasma simulations. A PIC code represents a plasma with computational particles, but the electromagnetic fields live on a grid. The particles and grid communicate through interpolation steps.

A simplified electrostatic PIC cycle looks like this:

1. **Deposit particle charge onto the grid:** particle positions are converted into a grid charge density $\rho(x,y)$.
2. **Solve a field equation on the grid:** for an electrostatic model, solve

   $$
   -\nabla^2 V = \frac{\rho}{\epsilon_0},
   \qquad
   \mathbf E=-\nabla V.
   $$

3. **Interpolate fields back to particles:** grid values of $\mathbf E$ are interpolated to particle positions.
4. **Push particles forward in time:** particle positions and velocities are updated using the Lorentz force.
5. **Repeat:** the new particle positions produce a new charge density, so the field solve is part of a time-dependent loop.

Today's finite-difference Poisson solver is therefore not a full PIC code, but it is a realistic subproblem inside one. The same ideas still matter: grid indexing, sparse operators, boundary conditions, residual checks, and convergence tests.

## Takeaways

- Electrostatic boundary-value problems provide a natural physics setting for finite differences.
- Conducting surfaces enter the calculation as boundary conditions on $V$.
- The finite-difference Laplacian becomes a sparse matrix acting on a 1D vector of grid unknowns.
- Each row of $A\mathbf u=\mathbf b$ is one finite-difference equation at one grid point.
- A small residual means the discrete system was solved; it does not prove the grid is fine enough.
- The electric field is derived from the potential, so field quality depends on both the potential solve and the numerical derivative.

## Open-Source References and Further Reading

- WarpX theory overview, including the PIC algorithm and electrostatic/Maxwell field solvers: https://warpx.readthedocs.io/en/latest/theory/intro.html
- Smilei documentation, an open-source electromagnetic PIC code for plasma simulation: https://smileipic.github.io/Smilei/
- eduPIC paper: an introductory 1D electrostatic PIC/MCC code and tutorial for plasma simulation: https://arxiv.org/abs/2103.09642
- *Fundamentals of Numerical Computation*, finite differences: https://fncbook.com/python/finitediffs/
- *Fundamentals of Numerical Computation*, differentiation matrices: https://fncbook.com/diffmats/
- SciPy sparse matrix documentation: https://docs.scipy.org/doc/scipy/reference/sparse.html
- SciPy sparse linear algebra documentation: https://docs.scipy.org/doc/scipy/reference/sparse.linalg.html
- MIT OpenCourseWare 8.07 Electromagnetism II: https://ocw.mit.edu/courses/8-07-electromagnetism-ii-fall-2012/

For Assignment 3, the important habit is to report **both** a convergence diagnostic and a physical/analytic comparison, not only the final field map.